In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# C2 2A: RGB8 quantization-only diagnostic

This notebook reads the fixed completed three-arm second-axis package and isolates only its `np.rint(clamp(rgb)*255).astype(uint8) / 255` stage. It does not generate a terminal, load a Wan pipeline or transformer, decode a latent, call FFmpeg, change colour space, or compress video. One frozen FP32 VAE is loaded once for three deterministic encodes.

In [ ]:
from pathlib import Path
import sys, subprocess
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'c2a-2a-colab-preparation'
SOURCE = Path('/content/c2a_quantization_diagnostic_source')
if SOURCE.exists():
    raise FileExistsError('Use a fresh runtime; preserve existing source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_BRANCH], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
print('Source:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
# Keep Colab CUDA PyTorch; actual versions are recorded by the result.


## Fixed source and fixed work

The sole input is `MyDrive/Video-WM/C2A_SecondAxis_Diagnostic/c2a_second_axis_20260914T145106Z`, using its persisted `ZERO`, `PLUS_E2`, and `MINUS_E2` float RGB tensors. The fixed C2 q reader remains group 2, channels 0/1, central 8x8 support, beta 0.25, rho 0.5, posterior mode and cache reset around each encode. Expected actual work is one VAE load and three VAE encodes; generation, transformer forwards, VAE decodes and FFmpeg calls are zero.

In [ ]:
from datetime import datetime, timezone
CONFIG = SOURCE / 'runtime/c2a/c2a_quantization_diagnostic_run.json'
RUN_ID = 'c2a_quantization_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/C2A_Quantization_Diagnostic') / RUN_ID
print(CONFIG.read_text())
print('Output:', OUTPUT)
if OUTPUT.exists():
    raise FileExistsError(str(OUTPUT))


In [ ]:
import os, signal
command = [sys.executable, '-m', 'runtime.c2a.run_quantization_diagnostic', '--config', str(CONFIG), '--output', str(OUTPUT), '--execute']
process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True)
try:
    returncode = process.wait()
except BaseException:
    try: process.send_signal(signal.SIGTERM)
    except ProcessLookupError: pass
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try: os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        process.wait()
    raise
print('launcher exit', returncode)
print((OUTPUT / 'result.json').read_text() if (OUTPUT / 'result.json').exists() else 'No result file')
if returncode: raise subprocess.CalledProcessError(returncode, command)


## Persisted result

The enabled run writes only to `MyDrive/Video-WM/C2A_Quantization_Diagnostic/<UTC-run-id>/` and never changes its input package. It saves each exact RGB8 uint8 tensor, source/pre-quantization q, quantized q, source/post-MP4 q, direct differences, O2/M2 vectors/norms, configuration, calls, and retained failures. Pixel error is labeled as pixel-only evidence and is not a q substitute.